# Linguistic Error Analysis

This notebook analyzes retrieval failures from a linguistic perspective.

Rather than treating retrieval as simply correct or incorrect, this experiment
asks why the semantic retrieval failed.

Research Question:

How do retrieval failures reflect linguistic differences between English,
German, and Russian?

The analysis focuses on linguistically motivated error categories:

- Morphological variation
- German compound representation
- Synonym variation
- Word order differences
- Semantic drift
- Language mismatch
- Out-of-vocabulary behavior
- Unknown retrieval errors

Pipeline:
```
Query
↓
Semantic Retrieval
↓
Retrieved Document
↓
Expected Document
↓
Linguistic Analyzer
↓
Error Category
↓
Error Collection
↓
Error Summary
```
This experiment provides the foundation for interpreting retrieval failures
rather than evaluating them only as numerical errors.

### Import Dependencies

In [2]:
import sys
from pathlib import Path

parent_dir = str(Path.cwd().parent)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import pandas as pd

from embeddings.encoder import MultilingualEncoder
from indexing.document_store import DocumentStore
from indexing.vector_index import VectorIndex
from retrieval.semantic_search import SemanticSearchEngine

from analysis.linguistic_analysis import LinguisticAnalyzer
from analysis.retrieval_errors import (
    RetrievalError,
    RetrievalErrorCategory,
    RetrievalErrorCollection
)

### Initialize the Semantic Retrieval Pipeline

In [3]:
# Maps multilingual text to a 512-dimensional vector space.
encoder = MultilingualEncoder()

print("Embedding model loaded.")

Loading embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Embedding model loaded.


### Create the Multilingual Ecaluation Collection

In [4]:
documents = [
    {
        "language": "English",
        "text": "The government introduced a new environmental policy.",
        "metadata": {
            "concept": "environment_policy"
        }
    },
    {
        "language": "German",
        "text": "Die Regierung führte eine neue Umweltpolitik ein.",
        "metadata": {
            "concept": "environment_policy"
        }
    },
    {
        "language": "Russian",
        "text": "Правительство ввело новую экологическую политику.",
        "metadata": {
            "concept": "environment_policy"
        }
    },
    {
        "language": "German",
        "text": "Das Unternehmen entwickelt erneuerbare Energietechnologien.",
        "metadata": {
            "concept": "renewable_energy"
        }
    },
    {
        "language": "Russian",
        "text": "Компания разрабатывает технологии возобновляемой энергии.",
        "metadata": {
            "concept": "renewable_energy"
        }
    },
    {
        "language": "German",
        "text": "Die Hausverwaltung bearbeitet die Anfrage.",
        "metadata": {
            "concept": "property_management"
        }
    },
    {
        "language": "German",
        "text": "Die Verwaltung des Hauses bearbeitet die Anfrage.",
        "metadata": {
            "concept": "property_management"
        }
    }
]


documents_df = pd.DataFrame(
    [
        {
            "language": document["language"],
            "text": document["text"],
            "concept": document["metadata"]["concept"]
        }
        for document in documents
    ]
)

documents_df

,language,text,concept
0,English,The government introduced a new environmental ...,environment_policy
1,German,Die Regierung führte eine neue Umweltpolitik ein.,environment_policy
2,Russian,Правительство ввело новую экологическую политику.,environment_policy
3,German,Das Unternehmen entwickelt erneuerbare Energie...,renewable_energy
4,Russian,Компания разрабатывает технологии возобновляем...,renewable_energy
5,German,Die Hausverwaltung bearbeitet die Anfrage.,property_management
6,German,Die Verwaltung des Hauses bearbeitet die Anfrage.,property_management


### Build the Document Store

In [5]:
# Store the original documents and their linguistic metadata.

document_store = DocumentStore()

for document in documents:
    document_store.add_document(
        text = document["text"],
        language = document["language"],
        metadata = document["metadata"]
    )

print(f"Stored documents: {len(document_store)}")

Stored documents: 7


### Generate Document Embeddings

In [6]:
# Convert every document into a multilingual semantic vector.

document_texts = [
    document["text"]
    for document in documents
]

document_embeddings = encoder.encode(document_texts)

print(f"Embedding shape: {document_embeddings.shape}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (7, 768)


### Build the FAISS Vector Index

In [7]:
# Create the vector index using the embedding dimensionality.

embedding_dimension = document_embeddings.shape[1]
vector_index = VectorIndex(embedding_dimension)
vector_index.add(document_embeddings)

print(f"Indexed vectors: {len(vector_index)}")

Indexed vectors: 7


### Initialize the Semantic Search Engine

In [9]:
# Combine the encoder, vector index, and document store
# into the complete semantic retrieval pipeline.

search_engine = SemanticSearchEngine(
    encoder = encoder,
    vector_index = vector_index,
    doc_store = document_store
)

print("Semantic search engine ready.")

Semantic search engine ready.


### Initialize the Linguistic Analyssis Components

In [10]:
# The analyzer categorizes retrieval differences using linguistically 
# motivated heuristics.

analyzer = LinguisticAnalyzer()
# The collection stores errors generated during the experiment.
error_collection = RetrievalErrorCollection()

print("Linguistic analysis components ready.")

Linguistic analysis components ready.


### Establish Evaluation Case

In [12]:
# Define a query and the document that represents the expected semantic result.

query = "The government created a new environmental policy."

expected_text = ("Die Regierung führte eine neue Umweltpolitik ein.")

print("Query:")
print(query)

print("\nExpected document:")
print(expected_text)

Query:
The government created a new environmental policy.

Expected document:
Die Regierung führte eine neue Umweltpolitik ein.


### Retrieve Candidate Documents

In [13]:
# Retrieve the highest-ranked documents for the query.

results = search_engine.search(
    query,
    top_k=5
)

retrieval_results = []

for result in results:
    retrieval_results.append(
        {
            "score": round(float(result.score), 4),
            "language": result.document.language,
            "text": result.document.text,
            "concept": result.document.metadata["concept"]
        }
    )

pd.DataFrame(retrieval_results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,score,language,text,concept
0,0.9858,German,Die Regierung führte eine neue Umweltpolitik ein.,environment_policy
1,0.9836,English,The government introduced a new environmental ...,environment_policy
2,0.9682,Russian,Правительство ввело новую экологическую политику.,environment_policy
3,0.3552,Russian,Компания разрабатывает технологии возобновляем...,renewable_energy
4,0.3390,German,Das Unternehmen entwickelt erneuerbare Energie...,renewable_energy


### Analyze Retrieved Results

In [14]:
# Compare each retrieved result with the expected document.
# Any identified linguistic errors are added to the collection.

for result in results:

    errors = analyzer.analyze(
        query = query,
        retrieved_result = result,
        expected_text = expected_text
    )

    for error in errors:
        error_collection.add_error(error)

print(f"Recorded errors: {error_collection.error_count()}")

Recorded errors: 4


### Inspect Recorder Errors

In [15]:
# Convert recorded errors into a readable table.

error_rows = []

for error in error_collection.all_errors():
    error_rows.append(
        {
            "category": error.category.value,
            "query": error.query,
            "retrieved_text": error.retrieved_text,
            "expected_text": error.expected_text,
            "explanation": error.explanation
        }
    )

errors_df = pd.DataFrame(error_rows)

errors_df

,category,query,retrieved_text,expected_text,explanation
0,unknown,The government created a new environmental pol...,The government introduced a new environmental ...,Die Regierung führte eine neue Umweltpolitik ein.,Retrieval differs from expectation but no ling...
1,morphological_variation,The government created a new environmental pol...,Правительство ввело новую экологическую политику.,Die Regierung führte eine neue Umweltpolitik ein.,Possible inflectional or morphological variati...
2,unknown,The government created a new environmental pol...,Компания разрабатывает технологии возобновляем...,Die Regierung führte eine neue Umweltpolitik ein.,Retrieval differs from expectation but no ling...
3,unknown,The government created a new environmental pol...,Das Unternehmen entwickelt erneuerbare Energie...,Die Regierung führte eine neue Umweltpolitik ein.,Retrieval differs from expectation but no ling...


### Analyze a German Compound Representation Difference

In [16]:
# Demonstrate the German compound heuristic directly.
# The two expressions represent related semantic content, but one uses a 
# compound noun while the other expresses the same concept with multiple words.

retrieved_text = "Die Hausverwaltung bearbeitet die Anfrage."
expected_text = "Die Verwaltung des Hauses bearbeitet die Anfrage."

compound_difference = analyzer.possible_compound_difference(
    retrieved_text,
    expected_text
)

print(f"Possible compound difference: {compound_difference}")

Possible compound difference: False


### Create a German Compound Retrieval Error

In [17]:
# Build a RetrievalError using the compound representation category.

compound_error = RetrievalError(
    category = RetrievalErrorCategory.COMPOUND_REPRESENTATION,
    explanation = (
        "The same concept is represented using a German compound "
        "versus a multi-word expression."
    ),
    query = "property management",
    retrieved_text = retrieved_text,
    expected_text = expected_text
)

error_collection.add_error(compound_error)

print("Compound representation error recorded.")

Compound representation error recorded.


### Analyze Morphological Variation

In [18]:
# Demonstrate the lightweight morphology heuristic.
# The current implementation identifies differences where the strings have the 
# same length but differ in form.

retrieved_text = "Haus"
expected_text = "haus"

morphology_difference = analyzer.possible_morphology_difference(
    retrieved_text,
    expected_text
)

print(f"Possible morphological difference: {morphology_difference}")

Possible morphological difference: False


### Record a Morphological Error

In [20]:
# Record the example using the project's morphological variation error category.

morphology_error = RetrievalError(
    category = RetrievalErrorCategory.MORPHOLOGICAL_VARIATION,
    explanation = (
        "The retrieved and expected forms differ morphologically "
        "while referring to the same lexical item."
    ),
    query = "house",
    retrieved_text = "Haus",
    expected_text = "haus"
)

error_collection.add_error(morphology_error)

print("Morphological variation error recorded.")

Morphological variation error recorded.


### Summarize Errors by Linguistic Category

In [22]:
# Count the recorded errors by linguistic category.

error_summary = error_collection.error_summary()

summary_rows = [
    {
        "category": category.value,
        "count": count
    }
    for category, count in error_summary.items()
]

summary_df = pd.DataFrame(summary_rows)

summary_df.sort_values(by = "count", ascending = False)

,category,count
0,unknown,3
1,morphological_variation,3
2,compound_representation,1


### Inspect Individual Error Categories

In [23]:
# Inspect all errors belonging to the compound representation category.

compound_errors = error_collection.by_category(
    RetrievalErrorCategory.COMPOUND_REPRESENTATION
)

for error in compound_errors:
    print("Category:", error.category.value)
    print("Query:", error.query)
    print("Retrieved:", error.retrieved_text)
    print("Expected:", error.expected_text)
    print("Explanation:", error.explanation)
    print()

Category: compound_representation
Query: property management
Retrieved: Die Hausverwaltung bearbeitet die Anfrage.
Expected: Die Verwaltung des Hauses bearbeitet die Anfrage.
Explanation: The same concept is represented using a German compound versus a multi-word expression.



### Linguistic Error Analysis Results

In [24]:
# Display the final collection of errors for inspection.

final_error_rows = []

for error in error_collection.all_errors():
    final_error_rows.append(
        {
            "Category": error.category.value,
            "Query": error.query,
            "Retrieved": error.retrieved_text,
            "Expected": error.expected_text,
            "Explanation": error.explanation
        }
    )

final_errors_df = pd.DataFrame(final_error_rows)

final_errors_df

,Category,Query,Retrieved,Expected,Explanation
0,unknown,The government created a new environmental pol...,The government introduced a new environmental ...,Die Regierung führte eine neue Umweltpolitik ein.,Retrieval differs from expectation but no ling...
1,morphological_variation,The government created a new environmental pol...,Правительство ввело новую экологическую политику.,Die Regierung führte eine neue Umweltpolitik ein.,Possible inflectional or morphological variati...
2,unknown,The government created a new environmental pol...,Компания разрабатывает технологии возобновляем...,Die Regierung führte eine neue Umweltpolitik ein.,Retrieval differs from expectation but no ling...
3,unknown,The government created a new environmental pol...,Das Unternehmen entwickelt erneuerbare Energie...,Die Regierung führte eine neue Umweltpolitik ein.,Retrieval differs from expectation but no ling...
4,compound_representation,property management,Die Hausverwaltung bearbeitet die Anfrage.,Die Verwaltung des Hauses bearbeitet die Anfrage.,The same concept is represented using a German...
5,morphological_variation,house,Haus,haus,The retrieved and expected forms differ morpho...
6,morphological_variation,house,Haus,haus,The retrieved and expected forms differ morpho...


### Interpreting this experiment

This notebook examines retrieval failure from a linguistic perspective rather than treating it as a simple binary outcome. It asks why a semantically related result may still be judged wrong, and whether the failure is caused by morphology, compounding, word order, lexical mismatch, or language-specific representation differences.

- If a retrieved result is close in meaning but differs in surface form, then the model may be capturing semantic similarity even when language-specific expression differs.
- If the model retrieves a nearby but incorrect concept, then the embedding space may be conflating related ideas rather than distinguishing them cleanly.
- If errors cluster around a linguistic category such as compound representation or morphology, then the failure pattern is not random; it reflects a systematic cross-lingual issue in the retrieval process.

The value of this analysis is that it moves beyond raw scores and explains the source of retrieval mistakes. In multilingual systems, linguistic structure is often the reason that a result fails even when it appears semantically plausible.

#### Cross-lingual interpretation

- English and German do not express the same ideas in the same way, so retrieval errors often arise from differences in surface form rather than actual concept mismatch.
- German compounds can compress meaning into a single token, while English often spreads it across multiple words.
- Morphological variation can change form without changing the core meaning, which may reduce apparent similarity in a naive retrieval system.
- The error categories in this notebook help interpret whether poor retrieval is caused by language-specific structure or by larger semantic drift.

### Connection to the relevant research questions in the README

This notebook connects directly to the project’s error-analysis and multilingual-meaning questions.

- Q1: Can multilingual embeddings align meaning across languages?
  - Retrieval mistakes reveal where alignment is weak, especially when related concepts are not mapped close enough in the shared embedding space.

- Q2: How does morphology affect semantic retrieval?
  - Morphological variation is a central theme here, since inflectional and compounding patterns can distort similarity across languages.

- Q3: Does linguistic preprocessing improve retrieval?
  - If preprocessing reduces the number of morphological or compound-related errors, then it is improving the semantic signal.

- Q4: How do retrieval failures reflect linguistic differences?
  - This notebook answers that question directly by classifying failure types according to linguistic structure, such as compounds, morphology, and word-order variation.

Overall, this experiment provides the explanatory layer behind the retrieval results: it shows that multilingual semantic search does not fail only because a candidate is wrong, but because different languages encode meaning through different linguistic mechanisms.